# LangGraph 조건부 분기를 이용한 동적 에이전트 워크플로우 설계

## 환경 설정

In [1]:
# 클래스 / 함수 임포트
from typing import Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END, add_messages

In [2]:
# ChatOpenAI 클래스 -> llm 생성
llm = ChatOpenAI(model="gpt-5-nano")

In [3]:
# TavilySearch 클래스 -> Tavily 검색 도구 생성 
search_tool = TavilySearch(max_results=3)

## 상태 정의

In [5]:
# Pydanctic 라이브러리의 BaseModel 클래스 상속 -> 커스텀 자료형 생성
class AgentState(BaseModel):
    # 상태 메시지 누적 구조 생성
    messages: Annotated[list[BaseMessage], add_messages] = []
    # 웹 검색 필요 여부에 대한 표시 (True: 검색 필요, False: 불필요)
    needs_search: bool = False
    # Tavily 검색 결과 저장
    search_results: str = ""

## 노드 함수 정의

In [6]:
# 사용자 질문 의도 분류 노드 함수 정의
"""
# 사용자의 질문을 분석 후 실시간 웹 검색이 필요한지를 단답형(SEARCH / NO_SEARCH)으로 생성
"""

def classifier_node(state: AgentState) -> dict:
    print("[Node 1: 질문 의도 분류] 실행 중...")
    print('-'*80)

    # 상태 state 출력
    print(f'state: {state}')
    print('-'*80)

    # 대화 스토리 -> 사용자 질문 추출 -> 상태 -> messages 
    user_question = state.messages[-1].content
    print(f"[사용자 질문 원본]: {user_question}")

    print('-'*80)

    # 프롬프트 생성: LLM에게 SEARCH 또는 NO_SEARCH 단어 하나만 반환하도록 명확하게 지시

    ## 시스템 프롬프트
    system_prompt="""
    당신은 사용자의 질문을 분석하는 전문가입니다.
    사용자의 질문에 답변하기 위해 실시간 웹 검색이 반드시 필요한지 판단하세요.

    - 최신 시사 뉴스, 실시간 정보, 특정 연도 실적(예: 2026년 실적) 등 최신 데이터 조회가 필요한 경우: 'SEARCH'
    - 일반 상식, 개념 설명, 프로그래밍 문법, 단순 인사말 등 모델 자체 지식으로 충분한 경우: 'NO_SEARCH'

    반드시 'SEARCH' 또는 'NO_SEARCH' 단어 하나만 정확하게 출력하세요. 다른 설명은 일체 포함하지 마세요.
    """    

    ## 전체 프롬프트 작성
    messages = [
        ("system",system_prompt),
        ("user", f"질문: {user_question}")
    ]
    response = llm.invoke(messages) 
    ## llm의 답변 -> response -> AIMessage(content="답변")
    print(f'llm의 답변: \n{response}')

    print('-'*80)

    ## llm의 답변 -> 텍스트('SEARCH ' 또는 'NO_SEARCH') 추출
    decision = response.content.strip().upper()

    ## 단답형 응답 문자열 -> 블리언 변환
    needs_search = (decision=="SEARCH")
    print(f"LLM 판단 결과: {decision} (검색 필요 여부: {needs_search})")

    ## needs_search 값을 반환 -> 상태 업데이트
    return {"needs_search": needs_search}

In [ ]:
# Tavily 웹 검색 노드 함수 정의
def tavily_search_node(state: AgentState) -> dict: 
    """
    # 기능: TavilySearch 도구를 실행하여 최신 웹 정보를 수집하고 텍스트로 가공한다.
    """
    print("[Node 2: Tavily 웹 검색 도구 실행] 실행 중...")
    print('-'*80)

    # 대화 히스토리 -> 사용자 최신 질문 추출 -> state.messages[-1] 
    query = state.messages[-1].content
    print(f"[검색 쿼리]: {query}")

    print('-'*80)

    # TavilySearch 검색 도구 실행
    try:
        # TavilySearch 검색 도구 실행 
        # 실행 결과: 딕셔너리 -> 키: results -> 값: 딕셔너리의 리스트 (검색 결과 3개: 딕셔너리 3개) 
        raw_results = search_tool.invoke({"query": query})

        ## 1. 도구 실행 결과 -> 딕셔너리 -> 'reuslts'키 -> 값: 리스트 추출, 없으면 빈 리스트
        if isinstance(raw_results, dict):
            results = raw_results.get("results", [])
        else:
            results = []

        ## 2. 리스트([{검색결과1}, ...]) -> 각 검색 결과(dict) -> 출처(url)와 본문(content) 추출
        context_parts = []
        for item in results:
            if isinstance(item, dict):
                url = item.get("url", "")
                content = item.get("content", "")
                context_parts.append(f"출처: {url}\n내용: {content}")
            else:
                context_parts.append(str(item)) 

        ## 3. 3개의 검색 결과 문자열(출처+본문) -> 하나의 문자열로 병합 -> "\n\n".join()
        if context_parts:
            combined_results = "\n\n".join(context_parts)
            print(f"웹 검색 성공 (총 {len(context_parts)}개 문서 수집 완료)")
        else:
            combined_results = "검색 결과가 없습니다."
            print("검색 결과가 비어 있습니다.")

    except Exception as e:
        error_message=f"검색 중 오류 발생: {e}"
        print(error_message)

    # 정제된 검색 결과(combined_results) 반환 -> LangGraph -> 상태 업데이트
    return {"search_results": combined_results}

In [11]:
# 최종 답변 생성 노드 함수 정의
def answer_node(state: AgentState) -> dict:
    """
    # 기능: 검색 결과 유무에 따라 시스템 프롬프트를 동적으로 변경하여 최적의 답변을 생성
    """
    print("[Node 3: 최종 답변 생성] 실행 중...")
    print('-'*80)

    # 대화 히스토리 -> 사용자 질문 추출 -> HumanMessage -> state.messages[-1]
    user_question = state.messages[-1].content

    # 검색을 거쳐온 경우와 거치지 않은 경우에 맞춰 시스템 프롬프트를 다르게 구성
    if state.needs_search and state.search_results:
        # 웹 검색 결과 -> 프롬프트에 반영
        system_prompt = f"""
        당신은 정확한 최신 정보를 제공하는 전문 AI 비서입니다.
        아래 제공된 [웹 검색 결과]를 바탕으로 사용자의 질문에 친절하고 명확하게 답변해 주세요.

        [웹 검색 결과]
        {state.search_results}
        """
    else:
        # 일반적인 질문인 경우: 일반적인 친절한 AI 비서로서의 지침 부여
        system_prompt = """당신은 친절하고 똑똑한 AI 비서입니다. 사용자의 질문에
        명쾌하게 답변해 주세요."""

    # 전체 프롬프트 작성
    messages=[
        ("system", system_prompt),
        ("user", user_question)
    ]

    # llm.invoke() 함수 호출 -> llm 답변 -> AIMessage(content="텍스트")
    response=llm.invoke(messages)
    # print(f"[최종 AI 답변 원본]: \n{response}")

    print('-'*80)

    # 생성된 답변을 AIMessage 형태로 전달(return) -> LangGraph 시스템 -> 상태 업데이트
    return {"messages":[AIMessage(content=response.content)]}

## 라우팅 함수 정의

In [8]:
# 기능: state의 'needs_search' 값을 읽고, classifier_node 다음에 실행할 노드의 이름을 반환

def route_by_search_need(state: AgentState) -> str:
    if state.needs_search:
        print("[Router] 최신 정보 검색이 필요하므로 'search_node'로 이동합니다.")
        return "search_node"  
    else:
        print("[Router] 검색이 불필요하므로 즉시 'answer_node'로 직행합니다.")
        return "answer_node"     

## 그래프 구성 

In [12]:
# '상태' 기반 빈 그래프(노드X, 엣지X) 생성 -> StateGraph 클래스 호출 
graph = StateGraph(AgentState) 

# 그래프에 노드 추가 -> StateGraph 클래스 -> 전용 함수 -> add_node 함수 호츨
graph.add_node("classifier_node", classifier_node)
graph.add_node("search_node", tavily_search_node)
graph.add_node("answer_node", answer_node)

# 시작점 설정: 질문이 들어오면 항상 분류 노드부터 시작 -> set_entry_point 함수 호출
graph.set_entry_point("classifier_node")

# 그래프에 엣지 추가

## 조건부 엣지 연결 -> add_conditional_edges() 함수 호출
"""
# classifier_node 실행 후, route_by_search_need 함수의 반환값에 따라 분기(edge 생성)
# 결과: classifier_node->search_node 또는 classifier_node->answer_node
"""
graph.add_conditional_edges(
    "classifier_node",
    route_by_search_need,
    {
        "search_node": "search_node",
        "answer_node": "answer_node"
    }
)

## 검색 노드 실행 후에는 최종 답변 생성 노드 이동
graph.add_edge("search_node", "answer_node")

## 최종 답변 생성이 끝나면 워크플로우 종료
graph.add_edge("answer_node", END)

In [14]:
# 그래프 컴파일 -> StateGraph 클래스 전용 함수 -> compile 함수 호출
agent=graph.compile()

## agent 실행 및 결과 확인

In [ ]:
print("=" * 80)
print("테스트 1: 일반 지식 질문 (웹 검색 불필요 -> 즉시 답변 직행 경로 검증)")
print("=" * 80)

# 사용자 질문 생성
test_query1 = "파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘."

# 초기 상태 생성
inital_state1 = AgentState(messages=[HumanMessage(content=test_query1)])

# agent 실행 -> agent.invoke() -> 최종 상태 생성(LangGraph 시스템: dict 자료형으로 변환)
final_state1 = agent.invoke(inital_state1)
ai_response_text=final_state1['messages'][-1].content
print(f'[최종 AI 답변]: \n{ai_response_text}')

테스트 1: 일반 지식 질문 (웹 검색 불필요 -> 즉시 답변 직행 경로 검증)
[Node 1: 질문 의도 분류] 실행 중...
--------------------------------------------------------------------------------
state: messages=[HumanMessage(content='파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘.', additional_kwargs={}, response_metadata={}, id='125c7c6e-0d65-4e10-a83b-b053ce867e94')] needs_search=False search_results=''
--------------------------------------------------------------------------------
[사용자 질문 원본]: 파이썬에서 리스트와 튜플의 가장 핵심적인 차이점을 한 문장으로 설명해줘.
--------------------------------------------------------------------------------
llm의 답변: 
content='NO_SEARCH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 174, 'total_tokens': 249, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, '

In [16]:
print("=" * 80)
print("테스트 2: 실시간 최신 정보 질문 (웹 검색 필요 -> 검색 -> 답변 경로 검증)")
print("=" * 80)

# 사용자 질문 생성
test_query2 = "Nvidia 2026년 2분기 영업 실적 요약해 줘."

# 초기 상태 생성
initial_state2 = AgentState(messages=[HumanMessage(content=test_query2)])

# agent 실행 -> invoke() 함수 호출 -> 최종 상태 생성
final_state2 = agent.invoke(initial_state2)
ai_response_text2 = final_state2['messages'][-1].content
print(f"[최종 AI 답변]:\n{ai_response_text2}")

테스트 2: 실시간 최신 정보 질문 (웹 검색 필요 -> 검색 -> 답변 경로 검증)
[Node 1: 질문 의도 분류] 실행 중...
--------------------------------------------------------------------------------
state: messages=[HumanMessage(content='Nvidia 2026년 2분기 영업 실적 요약해 줘.', additional_kwargs={}, response_metadata={}, id='aac3ce59-fa91-44a6-b56b-de0c3b913645')] needs_search=False search_results=''
--------------------------------------------------------------------------------
[사용자 질문 원본]: Nvidia 2026년 2분기 영업 실적 요약해 줘.
--------------------------------------------------------------------------------
llm의 답변: 
content='SEARCH' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 266, 'prompt_tokens': 167, 'total_tokens': 433, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None,